### Testing cross-validation strategies for i)

To asses the bias variance tradeoff one can do resampling. Both bootstrap and kfold resampling was attempted to decompose the MSE. As hyperparameters were trained using the kfold cross validation strategy, kfold resampling was chosen for MSE decompostion estimates. However, to be sure this was the right choice we compared bootstrap and kfold resampling.

In [ ]:
from pathlib import Path
import sys

# If the kernel is opened from the notebook folder:
project_root = Path.cwd().resolve().parent  # -> /home/.../Project_1


if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

FUCNCT_DIR = Path(__file__).resolve().parent.parent / "Code"   # Project_1/figs

from Code.general_functions import make_data, np, design_matrix, FIG_DIR
from Code.resampling_methods import cross_validation, grid_search, bootstrap_resampling, train_through_gridsearchCV, mse_decomposer
import Code.colors as c
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso

mindegree, maxdegree = 1, 21

degs = np.arange(mindegree, maxdegree)
x_,y_ = make_data()
x_train, x_test, y_train, y_test = train_test_split(x_,y_, test_size=0.2, random_state=2026)


# results, model_shorthands = train_through_gridsearchCV(x_, y_, models = ['ridge'], Ks=[5], return_std = True)
results, model_shorthands = train_through_gridsearchCV(x_, y_, models = ['ridge', 'lasso'], Ks=[5,10], return_std = True)

# decomposing mse with both kfold and bootstrap resampling
mse_decomposition_kfold = mse_decomposer(x_,y_, results, model_shorthands, resamples = 20)
mse_decomposition_bootstrap  = mse_decomposer(x_,y_, results, model_shorthands, resamples = 50, method = "bootstrap_resampling")


fig, axs = plt.subplots(1,1*len(model_shorthands), figsize = (5*len(model_shorthands), 5))   
# fig, axs = plt.subplots(2,2, figsize = (10, 10))   
# axs = axs.flat

for idx, model_shorthand in enumerate(model_shorthands):
        ax = np.atleast_1d(axs)[idx]  # handles the single-Axes case from plt.subplots
        
        # ax.errorbar(degs, results[model_shorthand]['score'], yerr = results[model_shorthand]['std_test_score'], color = color, linestyle = linestyle,  label = "test from CV")
        # for i, (mse_decomposition, std_decomposition) in enumerate(zip([mse_decomposition_kfold],[ std_decomposition_kfold])):
        for i, mse_decomposition in enumerate([mse_decomposition_bootstrap, mse_decomposition_kfold]):
                model ,_ ,_ ,k = model_shorthand.split()
               

                if i == 0:
                        resamplinglinestyle = "dotted"
                        label = 'bootstrap'
                elif i == 1:
                        resamplinglinestyle = 'solid'
                        label = 'kfold'

                if model == 'ridge':
                        if k == '5':
                                color = c.RIDGE
                        else:
                                color = c.RIDGE2
                elif model == 'lasso':
                        if k == '5':
                                color = c.LASSO
                        else:
                                color = c.LASSO2

                
                ax.plot(degs, mse_decomposition[model_shorthand]['error'],
                        label = f"{label} error", color = c.ERROR, linestyle = resamplinglinestyle)
                ax.plot(degs, mse_decomposition[model_shorthand]['variance'],
                        label = f"{label} variance", color = c.BIAS, linestyle = resamplinglinestyle)
                ax.plot(degs, mse_decomposition[model_shorthand]['bias2_plus_noise'], 
                        label = f"{label}" + r" bias  $\sigma^2$", color = c.VARIANCE, linestyle = resamplinglinestyle)
                
                ax.set_yscale("log")
                ax.set_title(model_shorthand)
                ax.set_xticks(degs)
                ax.grid(which="both", alpha=0.5)
                ax.legend()
                ax.set_title(f"{model.capitalize()} k = {int(k)}" )



fig.supxlabel("Polynomial Degree").set_in_layout(True)
# fig.supylabel("Cross-validated MSE").set_in_layout(True)
fig.tight_layout()
plt.savefig(fname=FIG_DIR/'Part_i_hyperparameters_msedecomposition.pdf')
